In [48]:
import numpy as np
import pandas as pd

In [49]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [50]:
df=pd.read_csv(r"C:\Users\tyagi\Downloads\train.csv")
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [51]:
df.drop(columns= ['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [52]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [53]:
x=df.iloc[:,1:]
x

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
886,2,male,27.0,0,0,13.0000,S
887,1,female,19.0,0,0,30.0000,S
888,3,female,NaN,1,2,23.4500,S
889,1,male,26.0,0,0,30.0000,C


In [54]:
y=df[['Survived']]

y

,Survived
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


In [55]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [56]:
x_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [57]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

### applying  imputer on age and embarked

In [58]:
si_age=SimpleImputer()
si_embarked=SimpleImputer(strategy='most_frequent')
x_train_age=si_age.fit_transform(x_train[['Age']])
x_train_embarked=si_embarked.fit_transform(x_train[['Embarked']])

x_test_age=si_age.transform(x_test[['Age']])
x_test_embarked=si_embarked.transform(x_test[['Embarked']])

### one hot encoding on  sex and embarked


In [59]:
ohe_sex=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_embarked=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
x_train_sex=ohe_sex.fit_transform(x_train[['Sex']])
x_train_embarked=ohe_embarked.fit_transform(x_train_embarked)

x_test_sex=ohe_sex.transform(x_test[['Sex']])
x_test_embarked=ohe_embarked.transform(x_test_embarked)

In [60]:
 x_train_sex.shape

(712, 2)

In [61]:
x_train_embarked.shape

(712, 3)

In [62]:
ohe_embarked.categories_

[array(['C', 'Q', 'S'], dtype=object)]

#### seprate out remaining columns like Pclass ,SibSp,Parch ,Fare from the x_train and x_test  data 

In [65]:
x_train_rem=x_train.drop(columns=['Sex','Age','Embarked'])
x_test_rem=x_test.drop(columns=['Sex','Age','Embarked'])
x_train_rem.shape

(712, 4)

In [66]:
x_train_transform=np.concatenate((x_train_rem,x_train_sex,x_train_embarked,x_train_age),axis=1)

In [67]:
x_train_transform.shape

(712, 10)

In [68]:
x_test_transform=np.concatenate((x_test_rem,x_test_sex,x_test_embarked,x_test_age),axis=1)

In [69]:
x_test_transform.shape

(179, 10)

In [71]:
clf=DecisionTreeClassifier()
clf.fit(x_train_transform,y_train)

DecisionTreeClassifier()

In [72]:
y_pred=clf.predict(x_test_transform)

In [74]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7821229050279329

In [75]:
import pickle

In [78]:
pickle.dump(ohe_sex, open('ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('ohe_embarked.pkl', 'wb'))
pickle.dump(clf, open('clf.pkl', 'wb'))